In [1]:
from qdrant_client import QdrantClient
from qdrant_client.http.models import PointStruct, VectorParams, Distance, QueryResponse
from ollama import Client as OllamaClient, ChatResponse
from typing import Mapping, Any, Optional, Sequence
import os
import hashlib

In [2]:
_ollama_client: Optional["OllamaClient"] = None

def _get_ollama_client() -> OllamaClient:
    global _ollama_client
    if _ollama_client is None:
        host = os.getenv("OLLAMA_HOST", "http://localhost:11434")
        _ollama_client = OllamaClient(host=host)
    return _ollama_client

In [3]:
_qdrant_client: Optional["QdrantClient"] = None

def _get_qdrant_client() -> QdrantClient:
    global _qdrant_client
    if _qdrant_client is None:
        url = os.getenv("QDRANT_HOST", "http://localhost:6333")
        _qdrant_client = QdrantClient(url=url)
    return _qdrant_client

In [4]:
def embed(text:str) -> list[float]:
    _ollama_client = _get_ollama_client()
    response: ChatResponse = _ollama_client.embed(model="nomic-embed-text", input=text) # type: ignore
    return response.embeddings[0] # type: ignore

In [5]:
def document_id(document: str, truncate: int = 32) -> str:
    normalized = " ".join(document.split()).strip().lower()
    encoded = normalized.encode('utf-8')
    hash = hashlib.sha256()
    hash.update(encoded)
    return hash.hexdigest()[:truncate]

In [6]:
def qdrant_write(document: str, collection: str) -> None:
    vector: list[float] = embed(document)
    point = PointStruct(
        id=document_id(document), #automatically translated to UUID
        vector=vector,
        payload={"text": document}
        )
    
    qdrant_client = _get_qdrant_client()

    if qdrant_client.collection_exists(collection_name=collection):
        pass
    else:
        qdrant_client.create_collection(
            collection_name=collection,
            vectors_config=VectorParams(
                size=len(vector),
                distance=Distance.COSINE
            )
        )

    qdrant_client.upsert(
        collection_name=collection,
        points=[point],
    )

In [10]:
qdrant_write("France is a country in Europe. Its capital is Paris.", "Trek")

qdrant_client = _get_qdrant_client()
qdrant_client.scroll(collection_name="Trek")

([Record(id='a284ca9d-2b50-ce30-098f-8c5d568a55cd', payload={'text': 'France is a country in Europe. Its capital is Paris.'}, vector=None, shard_key=None, order_value=None)],
 None)

In [8]:
#qdrant_client = _get_qdrant_client()
#qdrant_client.delete_collection(collection_name="test_collection")